# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Yeraly Adiat
**Student ID:** s23068138
**Group:** 9
**Repository:** https://github.com/adiateraly/hw3-adiateraly.git

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>I used Claude as a helper. It explained the task and the formulas in simple words, helped me set up VS Code.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

lowercase, split at spaces and hyphens, drop punctuation, no stop words removed.
N = 3, k1 = 1.5, b = 0.75.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

I counted the tokens of each document. D1 has 9, D2 has 10 (`Retrieval-Augmented` is split into two tokens), D3 has 6.

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 ≈ 8.333

**Query terms:** how, does, rag, retrieve, information (5 terms)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

I looked for each query term as an exact token in each document. D2 contains `retrieves` and `retrieval`, but these are different tokens from `retrieve`, so f(retrieve, D2) = 0. The other four query terms do not appear in any document.

IDF for every term is the same, because n(t) = 0 for all of them:
idf(t) = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7) ≈ 1.9459

**BM25 of each document:**

> Every part of the sum has f(t, d) in the top. All f are 0, so no term contributes. Every part is 0.
>
> score(D1) = idf(how)·0 + idf(does)·0 + idf(rag)·0 + idf(retrieve)·0 + idf(information)·0 = 0

> score(D2) = idf(how)·0 + idf(does)·0 + idf(rag)·0 + idf(retrieve)·0 + idf(information)·0 = 0

> score(D3) = idf(how)·0 + idf(does)·0 + idf(rag)·0 + idf(retrieve)·0 + idf(information)·0 = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

All three documents get 0, so there is a three-way tie. This is not a mistake. BM25 only matches exact words. The query has `retrieve`, but D2 has `retrieves` and `retrieval`.

## A1 (b) — BM25 with the stems from README §3

Stems: retrieve / retrieves / retrieval → `retriev`;
 generate / generating / generation → `gener`;
information → `inform`; does → `doe`.
All other words stay the same.

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

Stemming replaces words but does not remove any, so the lengths are the same as in (a).

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 ≈ 8.333

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

idf(retriev) = ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(2.5 / 1.5) ≈ 0.5108.
The other words have n(t) = 0, so idf = ln 7 ≈ 1.9459, but f = 0 everywhere, so they add nothing.

**BM25 of each document**, worked:

> Only `retriev` appears in the documents, and only in D2 (twice).
>
> score(q, D2) = idf(retriev) · f · (k1 + 1) / (f + k1 · (1 − b + b · |d| / avgdl))

> = 0.5108 · 2 · 2.5 / (2 + 1.5 · (0.25 + 0.75 · 10 / 8.333))

> = 2.554 / (2 + 1.5 · 1.15)

> = 2.554 / 3.725 ≈ 0.6857
>
> score(q, D1) = 0 (no query term appears in D1)

> score(q, D3) = 0 (no query term appears in D3)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In (a), no document wins: all three score 0 and tie. The query contains `retrieve`, but D2 only contains `retrieves` and `retrieval`. BM25 matches exact tokens, so these count as different words and nothing matches.
>
> In (b), D2 is first with 0.6857. After stemming, `retrieve`, `retrieves` and `retrieval` all become `retriev`, which appears twice in D2. D1 and D3 contain no query term, so they stay at 0.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 ≈ 0.9644

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124 | 0.78 / (0.9644 · 0.8124) = 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225 | 1.08 / (0.9644 · 1.1225) = 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307 | 0.34 / (0.9644 · 0.8307) = 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 (cosine 0.9977). Its vector points almost exactly in the same direction as the query vector. D1 is very close behind (0.9956), while D3 is far away (0.4244).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Not with A1 (a): BM25 gave three zeros and a tie, while cosine similarity gives a clear order D2 > D1 > D3. 

> With A1 (b), the first place is the same (D2), but BM25 still puts D1 and D3 in a tie at 0, while cosine similarity separates them: D1 is almost as close to the query as D2, and D3 is much lower.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents, not at retrieval in general.

> BM25 only counts shared words. The query "How does RAG retrieve information?" shares no exact token with D1 or D3, and in (a) not even with D2, because `retrieve` is not the same token as `retrieves` or `retrieval`. 

> Embeddings compare meaning instead: D2 is about retrieving documents to answer a question, so it is closest, and D1 (language models answering questions) is on a related topic, so it is also close. 
> D3 (embeddings as numerical vectors) is a different subject, so it is far. So BM25 can miss a document that has the right meaning but different word forms, and embeddings can rank related-but-not-matching texts as close even when they share no words.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [30]:
import math, re
import numpy as np

docs = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
query_a = "How does RAG retrieve information?"

def tokenize_a(s):
    return re.findall(r"[a-z0-9]+", s.lower().replace("-", " "))

N_a, K1, B = len(docs), 1.5, 0.75

def bm25_check(tokens, q):
    avgdl = sum(len(t) for t in tokens.values()) / N_a
    print("lengths:", {k: len(v) for k, v in tokens.items()}, "| avgdl =", round(avgdl, 3))
    for term in q:
        n = sum(term in t for t in tokens.values())
        print(f"  {term}: n = {n}, idf = {math.log((N_a - n + 0.5) / (n + 0.5)):.4f}")
    for name, t in tokens.items():
        score = 0
        for term in q:
            n = sum(term in x for x in tokens.values())
            idf = math.log((N_a - n + 0.5) / (n + 0.5))
            f = t.count(term)
            score += idf * f * (K1 + 1) / (f + K1 * (1 - B + B * len(t) / avgdl))
        print(name, "BM25 =", round(score, 4))

print("A1 (a)")
tok_a = {k: tokenize_a(v) for k, v in docs.items()}
bm25_check(tok_a, tokenize_a(query_a))

stem = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
        "generate": "gener", "generating": "gener", "generation": "gener",
        "information": "inform", "does": "doe"}
print("\nA1 (b)")
tok_s = {k: [stem.get(w, w) for w in v] for k, v in tok_a.items()}
q_s = [stem.get(w, w) for w in tokenize_a(query_a)]
print("stemmed query:", q_s)
bm25_check(tok_s, q_s)

print("\nA2")
Qv = np.array([0.8, 0.2, 0.5])
vecs = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
print("||Query|| =", round(np.linalg.norm(Qv), 4))
for name, v in vecs.items():
    v = np.array(v)
    dot = Qv @ v
    cos = dot / (np.linalg.norm(Qv) * np.linalg.norm(v))
    print(name, "dot =", round(dot, 4), "| ||D|| =", round(np.linalg.norm(v), 4), "| cosine =", round(cos, 4))

A1 (a)
lengths: {'D1': 9, 'D2': 10, 'D3': 6} | avgdl = 8.333
  how: n = 0, idf = 1.9459
  does: n = 0, idf = 1.9459
  rag: n = 0, idf = 1.9459
  retrieve: n = 0, idf = 1.9459
  information: n = 0, idf = 1.9459
D1 BM25 = 0.0
D2 BM25 = 0.0
D3 BM25 = 0.0

A1 (b)
stemmed query: ['how', 'doe', 'rag', 'retriev', 'inform']
lengths: {'D1': 9, 'D2': 10, 'D3': 6} | avgdl = 8.333
  how: n = 0, idf = 1.9459
  doe: n = 0, idf = 1.9459
  rag: n = 0, idf = 1.9459
  retriev: n = 1, idf = 0.5108
  inform: n = 0, idf = 1.9459
D1 BM25 = 0.0
D2 BM25 = 0.6857
D3 BM25 = 0.0

A2
||Query|| = 0.9644
D1 dot = 0.78 | ||D|| = 0.8124 | cosine = 0.9956
D2 dot = 1.08 | ||D|| = 1.1225 | cosine = 0.9977
D3 dot = 0.34 | ||D|| = 0.8307 | cosine = 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [43]:
import json, os, re
import numpy as np
from dotenv import load_dotenv

load_dotenv()   # reads OPENAI_API_KEY and HF_TOKEN from .env (in Colab: use Secrets)

corpus = [json.loads(line) for line in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]
qa = json.load(open("data/qa.json", encoding="utf-8"))["questions"]
ids = [c["id"] for c in corpus]

print("chunks:", len(corpus), "| fixed queries:", len(queries), "| questions:", len(qa))
print("OPENAI key found:", bool(os.getenv("OPENAI_API_KEY")), "| HF token found:", bool(os.getenv("HF_TOKEN")))

chunks: 74 | fixed queries: 8 | questions: 6
OPENAI key found: True | HF token found: True


## B1 — BM25 with `rank_bm25`

**My tokenisation:** I make all text lowercase. Then I split it with the regular expression `\w+`. 

This works for English, Kazakh and Russian letters. I do not use stemming. I do not remove stop words.

A code like `GR-305` becomes two tokens: `gr` and `305`. The index uses the `text` of each chunk (not the title), with `k1 = 1.5` and `b = 0.75`.

>

In [44]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in corpus], k1=1.5, b=0.75)

def bm25_scores(query_text):
    return np.array(bm25.get_scores(tokenize(query_text)))

def top_k(scores, k=3):
    return list(np.argsort(-scores, kind="stable")[:k])

def first_line(i, n=70):
    return corpus[i]["text"][:n].replace("\n", " ") + "..."

def show(name, scores, k=3):
    for rank, i in enumerate(top_k(scores, k), 1):
        print(f"  {name:9s} {rank}. {ids[i]:6s} {scores[i]:8.3f}  {first_line(i)}")

# quick test on one query
print(queries[0]["id"], queries[0]["text"])
show("BM25", bm25_scores(queries[0]["text"]))

Q-01 What is form GR-305 used for?
  BM25      1. GO-08    15.825  Form GR-305 is used to appeal a decision of the Grant Office. It is th...
  BM25      2. GO-12     8.195  A student who disagrees with a decision may appeal within 14 calendar ...
  BM25      3. GO-06     8.070  Form GR-104 is the application form for the need-based study grant. It...


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

I embed every chunk one time with `encode_document`. I embed every query with `encode_query`. Then I use cosine similarity (sklearn). The vectors are kept in a normal NumPy array in memory.

In [45]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))   # float32 (default)
doc_emb = model.encode_document([c["text"] for c in corpus], convert_to_numpy=True)
print("document vectors:", doc_emb.shape)

def emb_scores(query_text):
    q_emb = model.encode_query(query_text, convert_to_numpy=True)
    return cosine_similarity(q_emb.reshape(1, -1), doc_emb)[0]

print(queries[0]["id"], queries[0]["text"])
show("Embedding", emb_scores(queries[0]["text"]))

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 314.42it/s]


document vectors: (74, 768)
Q-01 What is form GR-305 used for?
  Embedding 1. GO-08     0.696  Form GR-305 is used to appeal a decision of the Grant Office. It is th...
  Embedding 2. GO-12     0.461  A student who disagrees with a decision may appeal within 14 calendar ...
  Embedding 3. GO-23     0.450  Если студент не согласен с решением, он может подать апелляцию по форм...


## My own queries

I wrote these two queries and their `relevant` chunks **before** I ran them.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What happens when the total support is over the 400,000 tenge cap? | GO-16 | **BM25.** The query has an exact number, "400" and "000", and the word "cap". These rare strings are in GO-16. BM25 can match them exactly. An embedding model can blur numbers. |
| Q-10 | Что делать, если я пропустил срок подачи заявки? (Russian: "What do I do if I missed the application deadline?") | GO-09, GO-14 | **Embeddings.** The chunks that answer it are in English (GO-09) and Kazakh (GO-14). The query is in Russian, so it has no words in common with them. An embedding model works across languages, so it can match the meaning. BM25 cannot. |

In [47]:
my_queries = [
    {"id": "Q-09", "text": "What happens when the total support is over the 400,000 tenge cap?",
     "relevant": ["GO-16"]},
    {"id": "Q-10", "text": "Что делать, если я пропустил срок подачи заявки?",
     "relevant": ["GO-09", "GO-14"]},
]
all_queries = queries + my_queries
print(len(all_queries), "queries in total")

10 queries in total


## Results for all ten or more queries: top 3 from each retriever

In [56]:
def hit(top_idx, relevant):
    return any(ids[i] in relevant for i in top_idx)

results = []
for q in all_queries:
    b, e = bm25_scores(q["text"]), emb_scores(q["text"])
    bt, et = top_k(b), top_k(e)
    results.append({"id": q["id"], "bm25_top": [ids[i] for i in bt], "bm25_hit": hit(bt, q["relevant"]),
                    "emb_top": [ids[i] for i in et], "emb_hit": hit(et, q["relevant"])})
    print(f'\n{q["id"]}: {q["text"]}   (relevant: {q["relevant"]})')
    show("BM25", b)
    show("Embedding", e)
    print(f'  hit@3  BM25: {results[-1]["bm25_hit"]} | Embedding: {results[-1]["emb_hit"]}')


mark = lambda x: "✓" if x else "✗"
print("\n| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |")
print("|---|---|:---:|---|:---:|")
for r in results:
    print(f'| {r["id"]} | {", ".join(r["bm25_top"])} | {mark(r["bm25_hit"])} | {", ".join(r["emb_top"])} | {mark(r["emb_hit"])} |')
bm25_total = sum(r["bm25_hit"] for r in results)
emb_total = sum(r["emb_hit"] for r in results)
print(f"| **hit@3 total** | | {bm25_total}/{len(results)} | | {emb_total}/{len(results)} |")

# helper: which queries are good examples for B3?
print("\nBoth hit      :", [r["id"] for r in results if r["bm25_hit"] and r["emb_hit"]])
print("BM25 only     :", [r["id"] for r in results if r["bm25_hit"] and not r["emb_hit"]])
print("Embedding only:", [r["id"] for r in results if r["emb_hit"] and not r["bm25_hit"]])
print("Both miss     :", [r["id"] for r in results if not r["bm25_hit"] and not r["emb_hit"]])


Q-01: What is form GR-305 used for?   (relevant: ['GO-08', 'GO-12', 'GO-23'])
  BM25      1. GO-08    15.825  Form GR-305 is used to appeal a decision of the Grant Office. It is th...
  BM25      2. GO-12     8.195  A student who disagrees with a decision may appeal within 14 calendar ...
  BM25      3. GO-06     8.070  Form GR-104 is the application form for the need-based study grant. It...
  Embedding 1. GO-08     0.696  Form GR-305 is used to appeal a decision of the Grant Office. It is th...
  Embedding 2. GO-12     0.461  A student who disagrees with a decision may appeal within 14 calendar ...
  Embedding 3. GO-23     0.450  Если студент не согласен с решением, он может подать апелляцию по форм...
  hit@3  BM25: True | Embedding: True

Q-02: My family is poor. Is there financial help for students like me?   (relevant: ['GO-01', 'GO-02', 'GO-03'])
  BM25      1. KZ-17     8.726  The lake currently covers about 16,400 km2 (6,300 sq mi). However, lik...
  BM25      2. GO-20     8.

In [58]:
from IPython.display import Markdown, display

mk = lambda x: "✓" if x else "✗"
lines = ["| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |",
         "|---|---|:---:|---|:---:|"]
for r in results:
    lines.append(f'| {r["id"]} | {", ".join(r["bm25_top"])} | {mk(r["bm25_hit"])} | {", ".join(r["emb_top"])} | {mk(r["emb_hit"])} |')
lines.append(f'| **hit@3 total** | | {sum(r["bm25_hit"] for r in results)}/{len(results)} | | {sum(r["emb_hit"] for r in results)}/{len(results)} |')
display(Markdown("\n".join(lines)))

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-16, GO-04, GO-03 | ✓ | GO-16, GO-03, GO-05 | ✓ |
| Q-10 | GO-23, GO-15, KZ-01 | ✗ | GO-11, GO-09, GO-19 | ✓ |
| **hit@3 total** | | 6/10 | | 10/10 |

## B3 — comparison

The table for all ten queries is printed in the code cell above.

**My own queries.** I expected BM25 to win Q-09 and embeddings to win Q-10. For Q-09 both retrievers found GO-16 first, so BM25 did not win alone. For Q-10 only the embeddings found a relevant chunk (GO-09, rank 2). BM25 missed it.

**Both retrievers agree:** Q-05 ("When was Astana renamed Nur-Sultan, and why?")

> Both retrievers gave the same top two chunks: KZ-05 first and KZ-06 second. It was easy because the query has rare words ("Astana", "renamed", "Nur-Sultan") and KZ-05 uses the same words. So the word match and the meaning match point to the same chunk. Q-01 was easy in the same way: both put GO-08 first.

**BM25 did better:** I did not find a query where BM25 hit and the embeddings missed. BM25 got hit@3 6/10 and the embeddings got 10/10.

> The nearest case is Q-01 ("What is form GR-305 used for?"). BM25 matched the exact token "305" and put GO-08 first with score 15.825. The second chunk had only 8.195, so the rare string gave a very strong signal. But BM25 also put GO-06 (the GR-104 form) at rank 3, because "form" and "gr" are in many chunks. The embeddings put GO-23 (the GR-305 appeal chunk in Russian) at rank 3. So exact matching of the code found the best chunk, but it did not stop a wrong form from coming in. For Q-09, BM25 matched the exact strings "400" and "000" (only in GO-16), and the embeddings found GO-16 too. The embedding model did not blur the number.

**Embeddings did better:** Q-07 ("Can the grant be sent to my mother's card?")

> The answer is in GO-15, which is in Russian. It says the money cannot go to the account of "родителей" (parents). BM25 returned GO-21 (score 14.652), GO-16 and GO-01, because the query has small common words like "can", "the" and "my". It has no word in common with GO-15: "mother" and "card" are not there, and the chunk is in another language. The embedding model put GO-15 first (0.422), because it connects "mother's card" with "account of parents" across two languages.
>
> Q-02 is another example. "poor" and "financial help" are synonyms of "low-income household" and "grant", and BM25 returned KZ-17 (a lake) and KZ-25 (beshbarmak), which are noise. The embeddings returned GO-01 first. Q-10 is a third example (a Russian query, answer chunks in English and Kazakh).

**Q-04 note:** For "What is the minimum GPA for the grant?" BM25 put GO-05 first (12.780). Its title is "Archived: the 2025 scheme (no longer in force)" and it says the GPA was 3.0. It has the same words as the query ("minimum", "GPA", "grant"). The right chunk, GO-02, was not in the BM25 top 3. The embeddings put GO-02 first (0.665) and GO-05 second (0.621). So a chunk with the same words can be old and wrong.


---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [49]:
from openai import OpenAI
import tiktoken

client = OpenAI()               # uses OPENAI_API_KEY from the environment
MODEL = "gpt-5.6-luna"
enc = tiktoken.get_encoding("o200k_base")

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def build_context(idx):
    return "\n\n".join(f"[{n}] {corpus[i]['text']}" for n, i in enumerate(idx, 1))

def build_messages(question, idx):
    user = f"Context:\n{build_context(idx)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def ask(messages):
    r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content, r.usage

def ask_plain(question):          # LLM only: no context, no system instruction about context
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}])
    return r.choices[0].message.content

def rag_answer(question, scorer, k=3):
    idx = top_k(scorer(question), k)
    answer, usage = ask(build_messages(question, idx))
    return answer, idx, usage

print("ready")

ready


## C1 — the complete prompt, printed before it is sent

I use question C-01 ("How much does a student in income band 2 receive, and when is it paid?"). I retrieve the top 3 with EmbeddingGemma. I use the system message from the README without changes

In [50]:
q = qa[0]                                     # C-01
idx = top_k(emb_scores(q["question"]), 3)
messages = build_messages(q["question"], idx)

print("Retrieved:", [(ids[i], round(float(emb_scores(q["question"])[i]), 3)) for i in idx])
print("\n" + "=" * 70 + "\nFULL PROMPT (exactly what the model receives)\n" + "=" * 70)
for m in messages:
    print(f"\n--- role: {m['role']} ---\n{m['content']}")
print("\n" + "=" * 70)

# count with tiktoken (o200k_base)
ctx = build_context(idx)
n_sys, n_q, n_ctx = len(enc.encode(SYSTEM)), len(enc.encode(q["question"])), len(enc.encode(ctx))
n_parts = n_sys + n_q + n_ctx
n_text = sum(len(enc.encode(m["content"])) for m in messages)     # full message text, with labels
print(f"System instructions : {n_sys}")
print(f"Question            : {n_q}")
print(f"Retrieved context   : {n_ctx}")
print(f"Total input (sum)   : {n_parts}")
print(f"(all message text together, with 'Context:' / 'Question:' labels and blank lines: {n_text})")

# send it and read usage
answer, usage = ask(messages)
print("\nusage.prompt_tokens :", usage.prompt_tokens)
print("difference (usage - sum of parts):", usage.prompt_tokens - n_parts)
print("\nAnswer:", answer)

# chunk statistics
tokens_per_chunk = [len(enc.encode(corpus[i]["text"])) for i in idx]
print("\nchunks:", len(idx), "| tokens per chunk:", tokens_per_chunk,
      "| context tokens:", n_ctx, "| context characters:", len(ctx))

Retrieved: [('GO-04', 0.585), ('GO-03', 0.484), ('GO-15', 0.451)]

FULL PROMPT (exactly what the model receives)

--- role: system ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

--- role: user ---
Context:
[1] The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. House

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 288 |
| **Total input** | 350 |
| **`usage.prompt_tokens` from the response** | 364 |

- Number of retrieved chunks: 3 (GO-04, GO-03, GO-15)
- Tokens per chunk: 90, 104, 85
- Total context size (tokens / characters): 288 / 1145

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The difference is 14 tokens (364 − 350). My total counts three separate parts: the system text, the question and the context. The API counts the whole request. The request has extra tokens that my three parts do not have: the labels "Context:" and "Question:", the new lines between the blocks, the numbers like [1], [2], [3], and the special tokens that the chat format adds to each message (for example the role marker and the start and end of the message). So `usage.prompt_tokens` is bigger. I saw the same kind of gap in HW1.

**What exactly does retrieval add to the model's input?**

> Retrieval adds only text. It adds three chunks that my search chose (GO-04, GO-03 and GO-15). They are pasted in the prompt as numbered blocks [1], [2] and [3]. This is 288 tokens (1145 characters), which is 288 of the 350 counted tokens, about 82%. Block [3] is in Russian, and it is just text in the same prompt. Nothing else is added: no document, no database and no search tool. The model reads this text like any other prompt and continues it. In this case the answer came from block [1].

## C2 — three systems, six questions

In [51]:
c2 = []
for item in qa:
    llm_only = ask_plain(item["question"])
    bm_ans, bm_idx, _ = rag_answer(item["question"], bm25_scores)
    em_ans, em_idx, _ = rag_answer(item["question"], emb_scores)
    c2.append({"id": item["id"], "llm": llm_only, "bm25": bm_ans, "emb": em_ans,
               "bm25_idx": bm_idx, "emb_idx": em_idx})
    print("=" * 78)
    print(item["id"], "|", item["question"])
    print("REFERENCE      :", item["reference"])
    print("LLM only       :", llm_only)
    print("BM25 RAG       :", bm_ans, "| chunks:", [ids[i] for i in bm_idx])
    print("Embedding RAG  :", em_ans, "| chunks:", [ids[i] for i in em_idx])

C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE      : 150,000 tenge for the academic year, in two equal instalments: October and February.
LLM only       : Which student funding scheme and academic year do you mean? “Income band 2” is used by several programs, and the amount and payment date vary by country and scheme.
BM25 RAG       : A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1] | chunks: ['GO-04', 'GO-02', 'GO-01']
Embedding RAG  : A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: **one in October and one in February**. [1] | chunks: ['GO-04', 'GO-03', 'GO-15']
C-02 | My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE      : 14 calendar days from the date on the decision letter, using form

In [59]:
from IPython.display import Markdown, display

def agrees(qid, a):
    a = a.lower().replace("*", "")
    if qid == "C-01": return "150,000" in a and "october" in a and "february" in a
    if qid == "C-02": return "14" in a and "gr-305" in a
    if qid == "C-03": return "2.67" in a
    if qid == "C-04": return "2017" in a and "2031" in a
    if qid == "C-05": return "1963" in a and "almaty institute of national economy" in a
    if qid == "C-06": return "do not know" in a or "don't know" in a

def short(a, n=60):
    a = a.replace("\n", " ").replace("|", "/").replace("*", "")
    return a[:n] + ("..." if len(a) > n else "")

keys = ("llm", "bm25", "emb")
c2_marks = {}
lines = ["| Question | LLM only | BM25 RAG | Embedding RAG |", "|---|---|---|---|"]
for r in c2:
    m = [bool(agrees(r["id"], r[k])) for k in keys]
    c2_marks[r["id"]] = m
    cells = " | ".join(f'{"✓" if x else "✗"} {short(r[k])}' for x, k in zip(m, keys))
    lines.append(f'| {r["id"]} | {cells} |')
c2_tot = [sum(v[i] for v in c2_marks.values()) for i in range(3)]
lines.append(f"| **agrees with `reference`** | {c2_tot[0]}/6 | {c2_tot[1]}/6 | {c2_tot[2]}/6 |")
display(Markdown("\n".join(lines)))

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Which student funding scheme and academic year do you mean? ... | ✓ A student in income band 2 receives 150,000 tenge for the ac... | ✓ A student in income band 2 receives 150,000 tenge for the ac... |
| C-02 | ✗ The deadline and form depend on the grant program and countr... | ✓ You have 14 calendar days from the date on the decision lett... | ✓ You have 14 calendar days from the date on the decision lett... |
| C-03 | ✗ It depends on the specific need-based grant. Many grants, su... | ✓ The minimum GPA is 2.67 on a 4.0 scale. [2] | ✓ The minimum GPA is 2.67 on the 4.0 scale. [1] |
| C-04 | ✓ Kazakhstan formally decided to replace Cyrillic with a Latin... | ✓ Kazakhstan decided to switch from Cyrillic to Latin in Octob... | ✓ Kazakhstan decided to switch from Cyrillic to Latin in Octob... |
| C-05 | ✗ Narxoz University was founded in 1963, originally as the Alm... | ✓ Narxoz University was founded in 1963 as the Almaty Institut... | ✓ Narxoz University was founded in 1963 as the Almaty Institut... |
| C-06 | ✗ The amount depends heavily on your country, university, prog... | ✓ I do not know how much you can get. The provided context say... | ✓ I do not know the amount. The need-based grant described is ... |
| **agrees with `reference`** | 1/6 | 6/6 | 6/6 |

The table above is printed by the code cell. I marked an answer ✓ when it has the key facts of the `reference` (for example "150,000", "October" and "February" for C-01; "14" and "GR-305" for C-02; "2.67" for C-03; "do not know" for C-06).

- **LLM only** could not answer the four questions about the fictional handbook (C-01, C-02, C-03, C-06). It asked me which scheme I mean, gave general advice, or listed US grants (Pell, FSEOG). It never gave the numbers from the handbook, and it did not invent a form code.
- **BM25 RAG** and **Embedding RAG** both answered the four handbook questions correctly. For C-06 both said that they do not know, which is the only correct answer.
- For C-05 both RAG systems said 1963 and "Almaty Institute of National Economy".

## C3 — question 1 of 3: C-03 (minimum GPA)

The printed output of the next cell shows items 1, 2 and 3.

**1. BM25 top 3:** GO-16 (15.8), GO-02 (14.06), GO-06 (13.85).
**2. EmbeddingGemma top 3:** GO-02 (0.629), GO-20 (0.569), GO-05 (0.545).
**3. Sent to the model:** BM25: blocks [1] GO-16, [2] GO-02, [3] GO-06. Embedding: blocks [1] GO-02, [2] GO-20, [3] GO-05 (the archived 2025 rule). The context blocks are printed above.

**4. How retrieval changed the answer:** more accurate and more grounded.

> LLM only did not know the handbook. It said "it depends" and talked about Pell grants and "around 2.0–3.0". Both RAG systems gave the exact value, 2.67 on the 4.0 scale, and cited the block with GO-02 ([2] for BM25 and [1] for embeddings).

**5. What was missed, and was it a retrieval problem or a generation problem?**

> Nothing was missed in the final answer: both RAG answers agree with the reference. But the retrieval was not clean. The embedding context had GO-05 at [3], which says "the minimum GPA for the grant was 3.0". This is the archived rule. So the retrieval put a wrong-rule chunk in the prompt. This was not a generation failure, because the model did not use it: it answered 2.67 and cited [1]. GO-05 says "ARCHIVED" and "no longer in force", and the model followed that. The BM25 context did not have GO-05 at all (GO-02 was at rank 2 with 14.06). So here retrieval was good enough in both cases, and the fact (2.67) was in the context both times. The risk was in the embedding context, and generation handled it.

In [52]:
def c3_report(qid):
    item = next(x for x in qa if x["id"] == qid)
    row = next(x for x in c2 if x["id"] == qid)
    qtext = item["question"]
    b, e = bm25_scores(qtext), emb_scores(qtext)
    print(qid, "|", qtext)
    print("relevant (from qa.json):", item["relevant"], "| reference:", item["reference"])
    print("\n1. BM25 top 3          :", [(ids[i], round(float(b[i]), 2)) for i in top_k(b)])
    print("2. EmbeddingGemma top 3:", [(ids[i], round(float(e[i]), 3)) for i in top_k(e)])
    for name, key in (("BM25 RAG", "bm25_idx"), ("Embedding RAG", "emb_idx")):
        print(f"\n3. Context sent to the model ({name}):")
        for n, i in enumerate(row[key], 1):
            print(f"   [{n}] {ids[i]} - {corpus[i]['title']}: {corpus[i]['text'][:160]}...")
    print("\nAnswers:")
    print("  LLM only      :", row["llm"])
    print("  BM25 RAG      :", row["bm25"])
    print("  Embedding RAG :", row["emb"])
    for name, key in (("BM25", "bm25_idx"), ("Embedding", "emb_idx")):
        got = [ids[i] for i in row[key]]
        print(f"  relevant chunks in {name} context:", [r for r in item["relevant"] if r in got],
              "| missing:", [r for r in item["relevant"] if r not in got])

c3_report("C-03")

C-03 | What is the minimum GPA needed to qualify for the need-based grant?
relevant (from qa.json): ['GO-02'] | reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

1. BM25 top 3          : [('GO-16', 15.8), ('GO-02', 14.06), ('GO-06', 13.85)]
2. EmbeddingGemma top 3: [('GO-02', 0.629), ('GO-20', 0.569), ('GO-05', 0.545)]

3. Context sent to the model (BM25 RAG):
   [1] GO-16 - Combining the grant with other support: The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all univ...
   [2] GO-02 - Who qualifies: An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; th...
   [3] GO-06 - Form GR-104: the application form: Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a let

## C3 — question 2 of 3: C-06 (master's student, no answer in the corpus)

The printed output of the next cell shows items 1, 2 and 3.

**1. BM25 top 3:** GO-16 (17.0), GO-01 (14.17), GO-20 (13.72).
**2. EmbeddingGemma top 3:** GO-20 (0.666), GO-16 (0.616), GO-01 (0.554).
**3. Sent to the model:** BM25: [1] GO-16, [2] GO-01, [3] GO-20. Embedding: [1] GO-20, [2] GO-16, [3] GO-01.

**4. How retrieval changed the answer:** more grounded.

> LLM only gave a long general answer about US grants, loans and assistantships. This is not an answer about this university. Both RAG systems said "I do not know". This is the only correct answer, because the handbook has no amount for master's students. They also used GO-20: BM25 cited [3] and the embeddings cited [1][3]. They said that master's students have a separate scheme and should ask their department.

**5. What was missed, and was it a retrieval problem or a generation problem?**

> Nothing was missed. The useful chunk GO-20 was in both contexts, at rank 3 for BM25 and rank 1 for the embeddings. It says that master's and doctoral students use a separate scheme that this handbook does not describe. The model used that fact and did not make up a number. So there was no retrieval failure and no generation failure. The system message ("If the answer cannot be found in the context, say that you do not know") did its job. It is interesting that BM25 ranked GO-20 only third (13.72, below GO-16 at 17.0). It was still in the top 3, so it was enough.

In [53]:
c3_report("C-06")

C-06 | I am a master's student. How much need-based grant can I get?
relevant (from qa.json): ['GO-20'] | reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

1. BM25 top 3          : [('GO-16', 17.0), ('GO-01', 14.17), ('GO-20', 13.72)]
2. EmbeddingGemma top 3: [('GO-20', 0.666), ('GO-16', 0.616), ('GO-01', 0.554)]

3. Context sent to the model (BM25 RAG):
   [1] GO-16 - Combining the grant with other support: The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all univ...
   [2] GO-01 - About the need-based study grant 2026: The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay...
   [3] GO-20 - Who cannot apply: The need-based grant is for full-time undergraduate students only. Part-time students, distance-l

## C3 — question 3 of 3: C-02 (appeal time and form)

The printed output of the next cell shows items 1, 2 and 3.

**1. BM25 top 3:** GO-05 (17.24), GO-08 (14.53), GO-12 (13.57).
**2. EmbeddingGemma top 3:** GO-08 (0.603), GO-12 (0.559), GO-11 (0.534).
**3. Sent to the model:** BM25: [1] GO-05 (archived 2025 scheme), [2] GO-08, [3] GO-12. Embedding: [1] GO-08, [2] GO-12, [3] GO-11 (missing document).

**4. How retrieval changed the answer:** more specific and more grounded.

> LLM only said that the deadline and form "depend on the grant program" and asked for more details. Both RAG systems gave the exact answer: 14 calendar days from the date on the decision letter, form GR-305. BM25 cited [3] (GO-12). The embedding system cited [1][2] (GO-08 and GO-12).

**5. What was missed, and was it a retrieval problem or a generation problem?**

> The answer was complete, but retrieval was not perfect, and this is the question where BM25 retrieved worse than I expected. BM25 gave the first place to GO-05 (17.24), the archived 2025 scheme, which has nothing to do with appeals. It had a higher score than GO-08 (14.53), the real appeal chunk. So one of the three slots was wasted. This is a retrieval problem (noise). It did not harm the answer, because GO-12 was in the context and it has both facts: "within 14 calendar days of the date on the decision letter, using form GR-305". GO-23 (the short Russian version of the same rule) was missing from both contexts. It was not needed, because GO-12 had the same facts. The embedding context also had one weak chunk, GO-11 (missing documents) at rank 3. In both systems the needed facts were in the context and the model used them, so there was no generation failure.

In [54]:
c3_report("C-02")

C-02 | My grant application was refused. How long do I have to appeal, and which form do I use?
relevant (from qa.json): ['GO-08', 'GO-12', 'GO-23'] | reference: 14 calendar days from the date on the decision letter, using form GR-305.

1. BM25 top 3          : [('GO-05', 17.24), ('GO-08', 14.53), ('GO-12', 13.57)]
2. EmbeddingGemma top 3: [('GO-08', 0.603), ('GO-12', 0.559), ('GO-11', 0.534)]

3. Context sent to the model (BM25 RAG):
   [1] GO-05 - Archived: the 2025 scheme (no longer in force): ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, a...
   [2] GO-08 - Form GR-305: appeals: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report...
   [3] GO-12 - How to appeal a decision: A student who disagrees with a decision may appeal within 14 calendar days of the date on th

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 counts matching words. It looks at how often a query word is in a chunk and how rare the word is in the corpus. An embedding model changes the text into a vector of numbers that shows the meaning, and then we compare vectors. So BM25 asks "are the words the same?" and embeddings ask "is the meaning similar?"

**2. When can BM25 perform better than embeddings?**

> BM25 can be better when the query has an exact rare string, like a code, a name or a number, and the chunk uses the same string. In my runs BM25 did not win any query by hit@3 (BM25 6/10, embeddings 10/10). But the exact match was strong: for Q-01 it matched "305" and gave GO-08 a score of 15.825 against 8.195 for the second chunk. BM25 is also simple and runs without a model.

**3. When can embeddings perform better than BM25?**

> Embeddings are better when the query uses different words from the chunk: a synonym, a paraphrase or another language. In my runs this was Q-07 ("mother's card" and the Russian chunk about the account of "родителей"), Q-02 ("poor" and "financial help" against "low-income" and "grant") and Q-10 (a Russian query, English and Kazakh chunks). BM25 missed all three.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity compares the direction of two vectors, not their length. Texts with similar meaning have vectors that point in a similar direction. A long text and a short text can have the same meaning but different vector lengths, so length should not change the score. Cosine gives a number from −1 to 1 that is easy to rank. In Part A, D2 got 0.9977 and D3 got 0.4244, so the ranking was clear.

**5. What information is actually sent to an LLM in a RAG system?**

> The model gets one prompt of text: the system instruction, the retrieved chunks as numbered blocks, and the user question. In C1 I printed it. It was 364 tokens by `usage.prompt_tokens`, and the three chunks were 288 tokens of it.

**6. Does the LLM search the original documents itself?**

> No. The LLM never opens a document. My code does the search with BM25 or EmbeddingGemma, picks the top chunks and pastes them into the prompt. The model only reads this text and continues it.

**7. What happens if retrieval returns an irrelevant chunk?**

> The model can use it, because it only sees the context. It wastes space and can lead to a wrong answer. In my runs, BM25 put the archived chunk GO-05 first for C-02 (17.24), above the real appeal chunk GO-08. The answer was still right, because GO-12 was also there. For C-03 the embedding context had GO-05 with the old GPA 3.0, but the model answered 2.67, because GO-05 says "no longer in force". A chunk with no such warning could give a wrong answer.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> A bigger K gives more chances to include the right chunk. It helps when the answer needs facts from several chunks. But a bigger K also adds more wrong or old chunks, more tokens and more cost, and it can confuse the model. I tested C-03 (minimum GPA) with the embedding retriever:
> - K = 1: chunk GO-02, 177 prompt tokens, answer: 2.67 [1].
> - K = 3: chunks GO-02, GO-20, GO-05, 326 prompt tokens, answer: 2.67 [1].
> - K = 8: chunks GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17, 757 prompt tokens, answer: 2.67 [1].
>
> The answer did not change, because GO-02 was already first. But the input grew from 177 to 757 tokens (more than four times), and K = 3 and K = 8 added the archived chunk GO-05. For this question K = 1 was enough, and a bigger K only added cost and risk.
>

In [55]:
item = next(x for x in qa if x["id"] == "C-03")
for K in (1, 3, 8):
    idx = top_k(emb_scores(item["question"]), K)
    answer, usage = ask(build_messages(item["question"], idx))
    print("=" * 70)
    print(f"K = {K} | chunks: {[ids[i] for i in idx]} | prompt tokens: {usage.prompt_tokens}")
    print("Answer:", answer)
print("\nReference:", item["reference"])

K = 1 | chunks: ['GO-02'] | prompt tokens: 177
Answer: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]
K = 3 | chunks: ['GO-02', 'GO-20', 'GO-05'] | prompt tokens: 326
Answer: The minimum GPA is **2.67 on a 4.0 scale**. [1]
K = 8 | chunks: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17'] | prompt tokens: 757
Answer: The minimum GPA is **2.67 on a 4.0 scale**. [1]

Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)


**9. Is a vector database required to build a RAG system?**

> No. In this notebook I stored the 74 vectors in a normal NumPy array (`doc_emb`, shape 74 × 768) in memory. I compared them with the query using `cosine_similarity` from sklearn. For 74 chunks this is fast. A vector database becomes useful when there are millions of chunks, when the data changes often, or when many users search at the same time.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?**

> Embeddings worked better for this corpus. The hit@3 over my ten queries was 6/10 for BM25 and 10/10 for EmbeddingGemma. BM25 missed Q-02, Q-04, Q-07 and Q-10: queries with synonyms (poor / low-income), another language (the Russian chunk GO-15 and my Russian query), or a query where the old chunk GO-05 had the same words as the question. In C2, the answers that agree with the reference were 6/6 for BM25 RAG and 6/6 for Embedding RAG (LLM only: 1/6). The two RAG systems were closer in C2 than in hit@3, because the top 3 of BM25 still held the needed chunk for those six questions, and the model used it.

---
# Conclusion

> In this homework I saw that a RAG system can fail in two places: in the search and in the model. When I looked at what was sent to the model, I could tell which one failed. In my runs the search made small mistakes (the archived chunk GO-05 in the top 3, a wasted slot for BM25 in C-02), but the model used the right chunk and ignored the old one. BM25 is good for exact words and codes, and embeddings are good for meaning, synonyms and other languages. Next I would build a **hybrid** system that uses BM25 and embeddings together and joins the two rankings. I would also remove or mark old documents like GO-05 so they cannot come back, and I would keep the instruction "say that you do not know", because C-06 shows why it is important. Then I would test with more questions and more values of K.